# Gold Biorecovery from E-Waste — Coupled Model

**One notebook over three models.** Crushed printed circuit boards in, recovered
metallic gold out.

| Part | Source document | Stage |
|---|---|---|
| I | `Bioleaching Models.docx` | Leaching: PCB → dissolved Au(S₂O₃)₂³⁻ |
| II | `GolB_model_writeup.docx` | Expression: GolB surface site density |
| III | `model.md` v3.0 | Capture → elution → reduction → recovery |

The three documents are reproduced verbatim in `docs/MODEL.md` and
`docs/MODEL.docx`. Nothing was corrected. Discrepancies found while combining
them are listed in `docs/ISSUES.md`.

This notebook holds narrative and figures only. Every equation lives in `src/`
and every number lives in `params/`. If you find yourself writing a `def` here,
it belongs in `src/`.

## Validation status — read first

`model.md` §0 opens with a statement that governs everything below:

> No part of this model has been checked against a measurement of this system.

That was written about Part III. It applies at least as strongly to Parts I and
II, whose parameter tables read *calibrate*, *unmeasured*, *fitted*, *assumed*,
*placeholder*.

**The deliverable is a ranked measurement priority list, not a yield prediction.**

## The chain, and the two joints

```
  crushed PCB
      │
      │  PART I — TetH → thiosulfate → Au/Cu leaching
      ▼
  leachate:  [Au(S2O3)2 3-],  [Cu(I)],  [Cu(II)],  [S2O3 2-]
      │
      │  ══ JOINT 1 ══  Au_feed        interfaces.au_feed_from_leach
      ▼
  PART III §2 — V1 capture on GolB-displaying E. coli
      ▲
      │  ══ JOINT 2 ══  q_max          interfaces.qmax_from_expression
      │
  PART II — mRNA → cytoplasmic protein → surface protein → active sites

  PART III continues:  capture → elution → reduction → recovery → eta_total
```

The three models are not three topics. Two quantities connect them, and each is
a quantity one document declares missing and another supplies.

**Joint 1 — `Au_feed`.** Part III §10: *"`Au_feed` has no upstream model though
it dominates V1."* Part I computes exactly that.

**Joint 2 — `q_max`.** Part III §9 ranks `q_max` measurement priority 2, noting
it is *"being addressed by the secretion model."* Part II computes the site
count that sets it.

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

from src import expression, interfaces, leach, params, recovery_chain

plt.rcParams.update({"figure.dpi": 110, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3})

P_EXPR = params.values(params.load("expression"))
P_CHAIN = params.load("recovery_chain")

print("parameter files:", [p.stem for p in sorted(params.PARAMS_DIR.glob("*.yaml"))])

### What is not measured

`model.md` §0 records a track record worth repeating before any number is read:

> Every parameter assumed without a source, then sourced, was wrong.

So the first thing the notebook prints is the list of parameters that are still
fitted, assumed, placeholder, or awaiting calibration.

In [ ]:
for part in ("leach", "expression", "recovery_chain"):
    flagged = params.unsourced(part)
    print(f"\n{part}  ({len(flagged)} not sourced)")
    for symbol, status in flagged:
        print(f"   {symbol:<28} {status[:70]}")

---

# Part I — Bioleaching

*Full text: `docs/01_bioleaching.md`. Code: `src/leach.py`.*

Bacteria express tetrathionate hydrolase (TetH), which breaks tetrathionate down
into thiosulfate. The thiosulfate leaches gold, and unavoidably copper, from
crushed PCB particles.

**Enzyme production**

$$\frac{d[E]}{dt} = k_{syn} X - (k_{deg} + \mu)[E]$$

**Tetrathionate**, Michaelis–Menten consumption plus Cu(II) regeneration:

$$\frac{d[S_4O_6^{2-}]}{dt} = F_{in} - \frac{k_{cat} E [S_4O_6^{2-}]}{K_m + [S_4O_6^{2-}]} + k_1 [Cu(II)][S_2O_3^{2-}]$$

**Thiosulfate**, production minus the parasitic Cu(II) loss and the metal draws:

$$\frac{d[S_2O_3^{2-}]}{dt} = \frac{k_{cat} E [S_4O_6^{2-}]}{K_m + [S_4O_6^{2-}]} - 2 k_1 [Cu(II)][S_2O_3^{2-}] - \frac{2}{V}R_{Au} - \frac{1}{V}R_{Cu}$$

**Gold**, Model B — oxygen and Cu(II) as parallel oxidants:

$$R_{Au} = A_{Au}(t)\,\frac{[S_2O_3^{2-}]}{K_S + [S_2O_3^{2-}]}\left\{ k_{Au,O_2}\frac{[O_2]}{K_{O_2}+[O_2]} + k_{Au,Cu}\frac{[Cu(II)]}{K_{Cu}+[Cu(II)]} \right\}$$

**The two geometries differ, and this is the interesting part of Part I.**
Gold on a PCB is a plated film, so it thins at constant face area:
$A_{Au} = m_{Au}/(\rho_{Au} t_{film})$, independent of how finely the board is
crushed. Copper is bulk metal and erodes inward, so
$A_{Cu}(t) = A_{Cu}(0)\,(n/n_0)^{2/3}$ and crush size does matter.

In [ ]:
# Geometry: gold area is set by plating thickness, not by crush size.
m_Au, rho_Au = 1e-4, 19300.0           # kg, kg/m3
for t_film in (0.05e-6, 0.5e-6, 1.0e-6):
    A = leach.gold_area(m_Au, rho_Au, t_film)
    print(f"t_film = {t_film*1e6:.2f} um  ->  A_Au = {A*1e4:7.2f} cm2")

print()
m_Cu, rho_Cu = 1e-2, 8960.0
for d in (2e-3, 4e-3, 6e-3):
    A = leach.copper_area_initial(m_Cu, rho_Cu, d)
    print(f"crush d = {d*1e3:.0f} mm    ->  A_Cu(0) = {A*1e4:7.2f} cm2")

### A demonstration run

**These rate constants are not the model's values.** Part I marks `k_cat`,
`k_Au,Cu`, `k_Cu,O2`, `k_Cu,Cu`, `k1`, `k3` and `k4` as *calibrate*, and
`params/leach.yaml` therefore carries them as `null`. The run below supplies
placeholder values so the machinery can be exercised. **Read the shapes, not the
numbers.**

Inventories are in µmol. See `docs/ISSUES.md` item 4 for why that choice matters.

In [ ]:
M_AU, M_CU = 197e-3, 63.55e-3          # kg/mol

demo = dict(
    k_syn=1e-12, X=1e12, k_deg=1.1e-3, mu=0.0,          # enzyme
    F_in=0.0, k_cat=10.0, K_m=500.0, k1=1e-5,           # tetrathionate
    V=1.0, O2=250.0,                                     # reactor
    m_Au=1e-4, rho_Au=19300.0, t_film=1e-6,              # gold
    k_Au_O2=11.0, k_Au_Cu=11.0e3, K_O2=50.0, K_S=5.0e4, K_Cu=100.0,
    m_Cu=1e-2, rho_Cu=8960.0, d=4e-3,                    # copper
    k_Cu_O2=1.0, k_Cu_Cu=1.0, k3=1e-5, k4=1e-6,
)
demo["A_Cu0"] = leach.copper_area_initial(demo["m_Cu"], demo["rho_Cu"], demo["d"])
demo["n_Cu0"] = demo["m_Cu"] / M_CU * 1e6                # umol

n_Au0 = demo["m_Au"] / M_AU * 1e6                        # umol
y0 = [0.0, 1e4, 1e5, n_Au0, 0.0, demo["n_Cu0"], 500.0, 500.0]

t, tr = leach.simulate(demo, y0, t_end=600.0, model="B")
print(f"integrated to t = {t[-1]:.0f} min of 600")
print(f"gold dissolved: {n_Au0 - tr['n_Au'][-1]:.2f} umol of {n_Au0:.1f} "
      f"({100*(1 - tr['n_Au'][-1]/n_Au0):.1f}%)")
print(f"leachate gold : {tr['Au_complex'][-1]:.3f} uM")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))

ax[0].plot(t, tr["E"] * 1e3)
ax[0].set(xlabel="time (min)", ylabel="[E]  (nmol/L)", title="(a) TetH")

ax[1].plot(t, tr["S4O6"], label="S$_4$O$_6^{2-}$")
ax[1].plot(t, tr["S2O3"], label="S$_2$O$_3^{2-}$")
ax[1].set(xlabel="time (min)", ylabel="umol/L", title="(b) sulfur species")
ax[1].legend()

ax[2].plot(t, tr["Au_complex"], label="Au(S$_2$O$_3$)$_2^{3-}$")
ax[2].set(xlabel="time (min)", ylabel="uM", title="(c) leachate gold")
ax[2].legend()

fig.suptitle("Part I demonstration run — placeholder rate constants", y=1.04)
fig.tight_layout()

**A structural warning visible in this run.** Copper is the reason Part I exists
in Model B form, and it is also the part that does not close. Starting from a
copper-free solution, Cu(II) has no production route and goes negative, which
then drags tetrathionate negative through the `k1` term. The run above avoids it
by starting with 500 µmol/L of each copper oxidation state. See
`docs/ISSUES.md` item 5 — this is a modelling question for the author of Part I,
and nothing was changed to hide it.

In [ ]:
# The same run started copper-free, showing the issue rather than avoiding it.
y0_free = [0.0, 1e4, 1e5, n_Au0, 0.0, demo["n_Cu0"], 0.0, 0.0]
t2, tr2 = leach.simulate(demo, y0_free, t_end=200.0, model="B")
print(f"Cu(II) minimum  : {tr2['Cu_II'].min():+.4f} umol/L")
print(f"S4O6 minimum    : {tr2['S4O6'].min():+.4f} umol/L")
print("Both are negative. ISSUES.md item 5.")

---

# Joint 1 — leachate gold becomes `Au_feed`

Part III §2.1 is unambiguous about what this number does: **`Au_feed` is the
dominant design variable**, the capture transition sits exactly at the capacity
ceiling $q_{max}X_{max}$ = 22 µM, and the recommended window is 10–20 µM.

Above 30 µM the `A_tox` disagreement costs 63–96% of capture. Dilution is
therefore a legitimate design lever, and the adapter reports where a given feed
lands rather than silently clipping it.

In [ ]:
au_feed = interfaces.au_feed_from_leach(tr["Au_complex"][-1], unit="umol/L")
print(f"Au_feed from Part I = {au_feed:.3f} uM")
print("guidance:", interfaces.feed_warning(au_feed) or "inside the 10-20 uM window")

print()
for feed in (3, 10, 15, 22, 25, 50, 300):
    note = interfaces.feed_warning(feed)
    print(f"  {feed:>4} uM  {'OK' if note is None else note[:88]}")

print()
cu = interfaces.cu_feed_from_leach(tr["Cu_I"][-1], tr["Cu_II"][-1])
print("copper carried into capture:", {k: round(v, 3) if isinstance(v, float) else v
                                        for k, v in cu.items()})
print("Part III has no copper term. Selectivity against Cu/Ag/Ni is unknown "
      "(model.md section 0).")

---

# Part II — GolB surface site density

*Full text: `docs/02_golb_expression.md`. Code: `src/expression.py`.*

Four pools in molecules per cell, each fed from upstream and drained by
degradation and by growth dilution:

$$m \;\rightarrow\; P_c \;\rightarrow\; P_s \;\rightarrow\; A = f P_s$$

$$\frac{dm}{dt} = \alpha N - (\delta_m + \mu)m$$
$$\frac{dP_c}{dt} = \beta m - (k_{sec} + \delta_c + \mu)P_c$$
$$\frac{dP_s}{dt} = k_{sec}P_c\left(1 - \frac{P_s}{P_{s,max}}\right) - (\delta_s + \mu)P_s$$

Dilution appears because a growing cell doubles its volume; even a perfectly
stable protein is diluted at the growth rate. For mRNA that term is small, for
stable protein it dominates.

In [ ]:
t_e, m, Pc, Ps, A = expression.simulate(P_EXPR, t_end=400.0)
ss = expression.steady_state(P_EXPR)
reported = params.load("expression")["reported_steady_state"]

print(f"{'':<8}{'ODE':>10}{'closed form':>14}{'reported':>10}")
for key, series in (("m", m), ("P_c", Pc), ("P_s", Ps), ("A", A)):
    print(f"{key:<8}{series[-1]:>10.1f}{ss[key]:>14.1f}{reported[key]:>10}")

print("\nP_s differs by 5.5% between the two routes: the closed form drops the")
print("saturation factor. The document's reported P_s follows the ODE and its")
print("reported A follows the closed form. See ISSUES.md item 3.")

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 6.4))

ax[0, 0].plot(t_e, m)
ax[0, 0].set(xlabel="time (min)", ylabel="mRNA / cell", title="(a) transcript dynamics")

ax[0, 1].plot(t_e, Pc, label="cytoplasmic $P_c$")
ax[0, 1].plot(t_e, Ps, label="surface $P_s$")
ax[0, 1].plot(t_e, A, label="active sites $A$")
ax[0, 1].set(xlabel="time (min)", ylabel="molecules / cell",
             title="(b) protein and active sites")
ax[0, 1].legend()

k_range = np.logspace(-3, 1, 200)
sites = [expression.steady_state({**P_EXPR, "k_sec": k})["A"] for k in k_range]
ax[1, 0].semilogx(k_range, sites)
ax[1, 0].plot(P_EXPR["k_sec"], ss["A"], "o", ms=7, label="operating point")
ax[1, 0].set(xlabel="$k_{sec}$  (min$^{-1}$)", ylabel="steady-state sites / cell",
             title="(c) sensitivity to display rate")
ax[1, 0].legend()

symbols = ["alpha", "N", "beta", "f", "k_sec", "delta_c", "delta_s", "delta_m", "mu"]
sens = [expression.normalised_sensitivity(P_EXPR, s) for s in symbols]
ax[1, 1].bar(range(len(symbols)), sens,
             color=["tab:green" if v > 0 else "tab:red" for v in sens])
ax[1, 1].set_xticks(range(len(symbols)))
ax[1, 1].set_xticklabels(symbols, rotation=45, ha="right")
ax[1, 1].axhline(0, color="k", lw=0.8)
ax[1, 1].set(ylabel="normalised sensitivity of $A^*$",
             title="(d) which parameter moves site density")

fig.tight_layout()

**Reading panel (d).** Promoter strength $\alpha$, copy number $N$, translation
strength $\beta$ and functional fraction $f$ all sit near +1: a 1% increase in
any of them gives about 1% more sites. These are the engineering handles.
$k_{sec}$ is only about +0.2, because it appears in both a numerator and a
denominator — which is the same story panel (c) tells. Growth rate $\mu$ is the
strongest negative influence, appearing in all three loss terms.

**The practical message is that making secretion faster yields diminishing
returns.** Raise transcription, translation, or copy number instead.

### Coupling to gold uptake

Each active site binds one Au(I) through a single Cys-X-X-Cys motif, so the
Langmuir plateau is the site count itself:

$$q(C) = \frac{A\,C}{K_d + C}$$

**Note the `K_d` conflict.** Part II uses 10 µM as an explicit placeholder;
Part III uses 0.1 µM. The half-saturation point of the figure below is `K_d` by
construction, so the whole x-axis shifts 100× depending on which is right.
`docs/ISSUES.md` item 1.

In [ ]:
C = np.logspace(-2, 3, 300)
A_star = ss["A"]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for K_d, panel, title in ((10.0, ax[0], "$K_d$ = 10 uM  (Part II)"),
                          (0.1, ax[1], "$K_d$ = 0.1 uM  (Part III)")):
    for mult, label in ((1, "base"), (2, "2x"), (5, "5x")):
        panel.semilogx(C, expression.langmuir_uptake(C, A_star * mult, K_d),
                       label=f"{label} ({A_star*mult:.0f} sites)")
    panel.axvline(K_d, color="k", ls=":", lw=1)
    panel.set(xlabel="leachate Au(I)  (uM)", title=title)
ax[0].set_ylabel("gold ions bound / cell")
ax[0].legend(fontsize=8)
fig.suptitle("Langmuir uptake — the same model under the two published $K_d$ values", y=1.05)
fig.tight_layout()

---

# Joint 2 — active sites become `q_max`

$$q_{max}\;\left[\frac{\mu M}{g/L}\right] = A\,[\text{sites/cell}] \times \frac{\text{cells}}{\text{g}} \times \frac{10^{6}}{N_A}$$

**This joint does not close, and the gap is the most important finding in this
repository.**

In [ ]:
q_from_II = interfaces.qmax_from_expression(A_star)
report = interfaces.qmax_disagreement(q_from_II)

print(f"Part II  ->  q_max = {report['part_II_uM_per_gL']:.4f} uM/(g/L)")
print(f"Part III     q_max = {report['part_III_uM_per_gL']:.4f} uM/(g/L)  (in use)")
print(f"Part III sweep low = 0.33 uM/(g/L)")
print(f"\nPart III is {1/report['ratio_II_over_III']:.0f}x higher than Part II,")
print(f"and {0.33/q_from_II:.0f}x higher than the bottom of its own sweep.")

X_max = P_CHAIN["V1_capture"]["Xmax"]["value"]
print(f"\ncapacity ceiling q_max * X_max at X_max = {X_max} g/L:")
print(f"   using Part III 4.4    ->  {4.4*X_max:8.2f} uM   (model.md section 2.1 quotes 22 uM)")
print(f"   using Part II  {q_from_II:.4f} ->  {q_from_II*X_max:8.4f} uM")
print("\nThe recommended 10-20 uM operating window sits two orders of magnitude")
print("above the second ceiling. ISSUES.md item 2 — resolve before quoting any")
print("coupled result.")

---

# Part III — capture, elution, reduction, recovery

*Full text: `docs/00_overview_gold_biorecovery.md`. Code: `src/recovery_chain.py`.*

$$\eta_{total} = \eta_{capture}\cdot\eta_{elute}\cdot\eta_{reduce}\cdot\eta_{phys}$$

**Critical path ≈ 42 h per batch.** Current result: `eta_total` = 10.5% median,
90% CI [0.0%, 49.4%].

`src/recovery_chain.py` is deliberately a shim. `model.md` names
`pipeline_model.py`, `r9L_fw.py` and `ensemble.py` as the working, verified code
for this part, and re-deriving them here would create a second version that
drifts from the first. Copy those three files into `src/` to complete the
module. The closed forms below need no ODE and run now.

### Elution — a ceiling and an approach, and they need different fixes

$$\eta_{elute}(t_k) = \underbrace{(1-\phi)}_{\text{ceiling: chemistry}}\underbrace{\left(1-e^{-k_{max}t_k}\right)}_{\text{approach: time}}$$

A short wash is cured by patience. A low ceiling is not. Sampling at 0.5/1/2/4 h
distinguishes them.

In [ ]:
phi = P_CHAIN["elution"]["phi"]["value"]
k_max = P_CHAIN["elution"]["k_max"]["value"]

t_k = np.linspace(0, 8, 400)
fig, ax = plt.subplots(figsize=(6, 3.4))
for k in (0.2, 0.5, k_max, 5.0):
    ax.plot(t_k, recovery_chain.eta_elute(t_k, phi, k),
            label=f"$k_{{max}}$ = {k} /h" + ("  (1 M TU)" if k == k_max else ""))
ax.axhline(1 - phi, color="k", ls="--", lw=1, label=f"ceiling $1-\\phi$ = {1-phi:.2f}")
ax.axvline(3 / k_max, color="grey", ls=":", lw=1, label="$3/k_{max}$")
ax.set(xlabel="elution time $t_k$ (h)", ylabel="$\\eta_{elute}$",
       title="Elution: ceiling is chemistry, approach is time")
ax.legend(fontsize=8)
fig.tight_layout()

print(f"eta_elute at the 6 h operating point: {recovery_chain.eta_elute(6.0, phi, k_max):.3f}")

### The electron budget — a ceiling that needs no rate constant

$$\eta_{reduce}\le\frac{2E_{tot}+2R_0}{n_{e,Au}A_0}$$

This is the cheapest useful calculation in the whole chain. It reproduces both
the current design point and the source papers' regime, and explains why those
papers never measured conversion.

In [ ]:
def budget(protein_g_per_L, A0_uM, n_e):
    E_tot = protein_g_per_L / 70000.0 * 1e6
    return recovery_chain.electron_budget_ceiling(E_tot, 0.0, A0_uM, n_e)

print(f"our design:      0.2 g/L protein, 5.78 uM Au(I)   -> {budget(0.2, 5.78, 1):>8.1%}")
print(f"source papers:   0.1 g/L protein, 2000 uM Au(III) -> {budget(0.1, 2000.0, 3):>8.4%}")
print("\nmodel.md section 4.1 quotes 98.9% and 0.048%.")

### Where the losses are

At Optimum A — 10 µM feed, $\phi$ = 0.13, $\eta_{phys}$ = 0.465:

| stage | efficiency | points lost |
|---|---|---|
| **recovery** | 46.5% | **53.5** |
| capture | 79.9% | 20.1 |
| elution | 87.0% | 13.0 |
| reduction | 100.0% | 0.0 |

**Recovery is the largest loss and the least-modelled stage.** It is also
contingent on particle size, which nobody has measured: if particles are larger
than 200 nm the ranking reverts to capture-first.

That contingency is a genuine coupling, not a caveat. Higher protein loading
improves conversion but makes smaller particles, which centrifuge worse. The two
efficiencies pull against each other and must be optimised as a product.

In [ ]:
sizes = {"10 nm (only sourced value)": 0.50, "50 nm": 0.80, ">200 nm": 0.97}
eta_wash = P_CHAIN["recovery"]["eta_wash"]["value"]
eta_ash = P_CHAIN["recovery"]["eta_ash"]["value"]

eta_capture, eta_reduce = 0.799, 1.00
eta_el = recovery_chain.eta_elute(6.0, phi, k_max)

print(f"{'particle size':<28}{'eta_phys':>10}{'eta_total':>12}")
for label, spin in sizes.items():
    ph = recovery_chain.eta_phys(spin, eta_wash, eta_ash)
    tot = recovery_chain.eta_total(eta_capture, eta_el, eta_reduce, ph)
    print(f"{label:<28}{ph:>10.3f}{tot:>12.1%}")

print(f"\neta_wash = {eta_wash} is unsourced and an upper bound (model.md section 5).")
print("Every number in this table is therefore optimistic by an unknown factor.")

---

# What to measure, in order

From `model.md` §9, now with Parts I and II folded in. The ranking is unchanged
at the top: the reduction time course is still the only test of whether the
chemistry works at all.

1. **Reduction time course** — gold + lysate, sample 0/1/2/4/8/24 h, read A₅₂₀.
   Addresses `k1` and `k2`, the joint #1 driver. Tests model *structure*: a
   nucleation-limited process shows an induction period then rapid conversion,
   an electron-transfer-limited one is fastest at t=0. **Opposite shapes in the
   first four hours.** This is also the only route to closing the 0.0% floor on
   `eta_total`.
2. **`q_max`** — display density. Parts II and III disagree by 255×
   (`ISSUES.md` item 2). Until this is settled the coupled chain has no
   defensible capture ceiling.
3. **Conversion quantification** — ICP-MS or colorimetric. Never measured by
   anyone, in any version of this chemistry.
4. **`eta_spin` and particle size** on real product. Recovery is the largest
   single loss and the least modelled.
5. **Eluent confirmation** — thiourea or acid-only. `k_max` is a fitted rate
   that silently absorbs "the eluent does not work."

Two items enter from Part I and belong on the same list:

6. **Cu(II) in the real leachate** — Part I makes Cu(II) the fast gold oxidant
   (≈10³× the oxygen route) and Part III has no copper term at all. Selectivity
   against Cu/Ag/Ni is unknown.
7. **`k_cat` for TetH** — unmeasured, and it sets the whole thiosulfate supply.

## What the model still cannot tell you

**Wet-lab facts:** whether the construct reduces gold in lysate at all; what
fraction converts; selectivity in real leachate; whether GolB's Cys10/Cys13
oxidise shut during a 13 h aerobic run.

**Structural gaps:** conditioning, the V1→V2 transfer, is assumed lossless;
particle-size distribution is absent though it couples two η terms; `k_dam` and
`A_tox` are structurally non-identifiable; Part I's Cu(II) balance does not close
from a copper-free start.

`Au_feed` is no longer on that list. Part I supplies it — which is the point of
putting these three documents together.